# BrainTart v2 - BraTS 2026 3D Inpainting

This notebook clones the BrainTart repository, downloads the challenge data,
trains the Attention U-Net 3D model with **v2 improvements**, runs MC-Dropout
ensemble inference, evaluates locally, and uploads to Synapse.

**v2 Improvements:**
1. **Edge Loss** - 3D Sobel gradient matching at tumor boundaries
2. **Frequency Loss** - FFT magnitude L1 for global texture coherence
3. **MC-Dropout Ensemble** - Stochastic inference for uncertainty-aware predictions
4. **Warmup Scheduler** - Linear warmup before cosine decay
5. **Gradient Accumulation** - Larger effective batch size on limited VRAM
6. **Test-Time Augmentation** - Axis-flip averaging at inference

**Hardware target:** 2x T4 (16 GB each) on Kaggle, PyTorch DDP.

### Sections
1. Environment Setup & Repo Clone
2. Synapse Auth & Data Download
3. Configuration Overview
4. Training (v2)
5. Inference - MC-Dropout Ensemble
6. Local Evaluation
7. Submission Checklist & Upload

## 1. Environment Setup & Repo Clone

In [ ]:
import subprocess, sys, os

# Clone the repo
REPO_URL = "https://github.com/PaulOkwija/BrainTart"
REPO_DIR = "/kaggle/working/BrainTart"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"Cloned BrainTart to {REPO_DIR}")
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
    print(f"Updated BrainTart at {REPO_DIR}")

# Install Python dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r",
     os.path.join(REPO_DIR, "requirements.txt")]
)
# synapseclient for data download + submission upload
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "synapseclient"]
)
print("Dependencies installed.")



In [ ]:
# Add repo to sys.path so imports work
sys.path.insert(0, REPO_DIR)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {props.name}  {props.total_memory/1e9:.1f} GB")



## 2. Synapse Auth & Data Download

In [ ]:
import zipfile
from pathlib import Path
import synapseclient
from kaggle_secrets import UserSecretsClient

SYNAPSE_TOKEN = UserSecretsClient().get_secret("brats")
syn = synapseclient.Synapse()
syn.login(authToken=SYNAPSE_TOKEN)
print("Logged in to Synapse.")

TRAINING_ENTITY   = "syn51523038"
VALIDATION_ENTITY = "syn51684975"

DOWNLOAD_DIR = Path("/kaggle/working/brats_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)



In [ ]:
# Download and extract TRAINING data
TRAIN_ROOT = Path("/kaggle/working/brats_data_train")
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

if not any(TRAIN_ROOT.rglob("BraTS-GLI-*-*")):
    print("Downloading training data...")
    entity = syn.get(entity=TRAINING_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))
    print(f"Downloaded to: {entity.path}")
    print("Extracting (this may take a while)...")
    with zipfile.ZipFile(entity.path, "r") as zf:
        members = zf.namelist()
        from tqdm.auto import tqdm
        for member in tqdm(members, desc=f"Extracting to { TRAIN_ROOT }"):
            zf.extract(member, path=TRAIN_ROOT)
    candidates = list(TRAIN_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        TRAIN_ROOT = candidates[0].parent
else:
    candidates = list(TRAIN_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        TRAIN_ROOT = candidates[0].parent
    print("Training data already extracted.")

train_cases = [p for p in TRAIN_ROOT.iterdir() if p.is_dir()]
print(f"TRAIN_ROOT   : {TRAIN_ROOT}")
print(f"Train cases  : {len(train_cases)}")



In [ ]:
# Download and extract VALIDATION data
VAL_ROOT = Path("/kaggle/working/brats_data_val")
VAL_ROOT.mkdir(parents=True, exist_ok=True)

if not any(VAL_ROOT.rglob("BraTS-GLI-*-*")):
    print("Downloading validation data...")
    try:
        entity = syn.get(entity=VALIDATION_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))
        print(f"Downloaded to: {entity.path}")
        print("Extracting (this may take a while)...")
        with zipfile.ZipFile(entity.path, "r") as zf:
            members = zf.namelist()
            from tqdm.auto import tqdm
            for member in tqdm(members, desc=f"Extracting to { VAL_ROOT }"):
                zf.extract(member, path=VAL_ROOT)
        candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
        if candidates:
            VAL_ROOT = candidates[0].parent
        val_cases = [p for p in VAL_ROOT.iterdir() if p.is_dir()]
        print(f"VAL_ROOT     : {VAL_ROOT}")
        print(f"Val cases    : {len(val_cases)}")
    except Exception as e:
        print(f"Could not download validation data: {e}")
        print("Falling back to training data for inference demo.")
        VAL_ROOT = TRAIN_ROOT
else:
    candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        VAL_ROOT = candidates[0].parent
    val_cases = [p for p in VAL_ROOT.iterdir() if p.is_dir()]
    print("Validation data already extracted.")
    print(f"VAL_ROOT     : {VAL_ROOT}")
    print(f"Val cases    : {len(val_cases)}")



In [ ]:
# Clean up downloaded zips to free disk space
import shutil
shutil.rmtree("/kaggle/working/brats_download", ignore_errors=True)



## 3. Configuration Overview

The `Config` dataclass now includes v2 hyperparameters. Override any defaults below.

In [ ]:
from configs import Config

DATASET_ROOT = str(TRAIN_ROOT)

cfg = Config()
cfg.DATASET_ROOT = DATASET_ROOT
cfg.makedirs()

# -- v2 overrides (uncomment to change) ---------------------------------------
# cfg.NUM_EPOCHS       = 100
# cfg.BATCH_PER_GPU    = 1      # reduce if OOM
# cfg.DROPOUT_RATE     = 0.15   # MC-dropout (0 = disabled)
# cfg.WARMUP_EPOCHS    = 3      # linear warmup before cosine
# cfg.GRAD_ACCUM_STEPS = 2      # effective batch = batch × accum × gpus
# cfg.LAMBDA_EDGE      = 0.25   # 3D Sobel edge loss
# cfg.LAMBDA_FREQ      = 0.1    # FFT magnitude loss

n_gpus = torch.cuda.device_count()
eff_batch = cfg.BATCH_PER_GPU * cfg.GRAD_ACCUM_STEPS * max(n_gpus, 1)

print("=" * 60)
print("BrainTart v2 Configuration")
print("=" * 60)
print(f"  Crop shape       : {cfg.CROP_SHAPE}")
print(f"  Base channels    : {cfg.BASE_CHANNELS}")
print(f"  Depth            : {cfg.DEPTH}")
print(f"  Dropout rate     : {cfg.DROPOUT_RATE}")
print(f"  Batch per GPU    : {cfg.BATCH_PER_GPU}")
print(f"  Grad accum steps : {cfg.GRAD_ACCUM_STEPS}")
print(f"  Effective batch  : {eff_batch}")
print(f"  Warmup epochs    : {cfg.WARMUP_EPOCHS}")
print(f"  Epochs           : {cfg.NUM_EPOCHS}")
print(f"  Loss weights     : L1={cfg.LAMBDA_L1} SSIM={cfg.LAMBDA_SSIM} "
      f"Edge={cfg.LAMBDA_EDGE} Freq={cfg.LAMBDA_FREQ}")
print(f"  MC samples       : {cfg.MC_SAMPLES}")
print(f"  TTA flips        : {cfg.USE_TTA_FLIPS}")
print("=" * 60)



## 4. Training (v2)

Runs DDP on available GPUs via the `train.py` script.

**v2 CLI args:** `--dropout`, `--warmup`, `--accum`, `--lam-edge`, `--lam-freq`

In [ ]:
import subprocess, sys

train_cmd = [
    sys.executable, f"{REPO_DIR}/train.py",
    "--dataset",        str(DATASET_ROOT),
    "--epochs",         str(cfg.NUM_EPOCHS),
    "--batch",          str(cfg.BATCH_PER_GPU),
    "--lr",             str(cfg.LR),
    "--seed",           str(cfg.SEED),
    "--crop",           str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
    "--base-ch",        str(cfg.BASE_CHANNELS),
    "--depth",          str(cfg.DEPTH),
    "--checkpoint-dir", str(cfg.CHECKPOINT_DIR),
    "--results-dir",    str(cfg.RESULTS_DIR),
    "--output-dir",     str(cfg.OUTPUT_DIR),
    "--mono-stages",    str(cfg.MONO_STAGES),
    "--mono-scales",    str(cfg.MONO_SCALES),
    # v2 args
    "--dropout",        str(cfg.DROPOUT_RATE),
    "--warmup",         str(cfg.WARMUP_EPOCHS),
    "--accum",          str(cfg.GRAD_ACCUM_STEPS),
    "--lam-edge",       str(cfg.LAMBDA_EDGE),
    "--lam-freq",       str(cfg.LAMBDA_FREQ),
]

print("Running:", " ".join(train_cmd))
result = subprocess.run(train_cmd, capture_output=False)
print(f"Training exited with code {result.returncode}")



In [ ]:
# Display the training curve
from IPython.display import Image, display

curve_path = cfg.OUTPUT_DIR / "training_curve_final.png"
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    curve_path = cfg.OUTPUT_DIR / "training_curve.png"
    if curve_path.exists():
        display(Image(filename=str(curve_path)))
    else:
        print("Training curve not found - check training output above.")



## 5. Inference - MC-Dropout Ensemble

Runs `inference.py` with MC-dropout ensemble + test-time flip augmentation.

Output format: `BraTS-GLI-XXXXX-YYY-t1n-inference.nii.gz` (240×240×155)

In [ ]:
import subprocess, sys

checkpoint = cfg.CHECKPOINT_DIR / "best_model.pt"
if not checkpoint.exists():
    # Fallback to latest periodic checkpoint
    ckpts = sorted(cfg.CHECKPOINT_DIR.glob("ckpt_epoch_*.pt"))
    checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    infer_cmd = [
        sys.executable, f"{REPO_DIR}/inference.py",
        "--dataset",       str(VAL_ROOT),
        "--checkpoint",    str(checkpoint),
        "--results-dir",   str(cfg.RESULTS_DIR),
        "--crop",          str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
        "--base-ch",       str(cfg.BASE_CHANNELS),
        "--depth",         str(cfg.DEPTH),
        "--mono-stages",   str(cfg.MONO_STAGES),
        "--mono-scales",   str(cfg.MONO_SCALES),
        # v2 args: MC-dropout + TTA
        "--dropout",       str(cfg.DROPOUT_RATE),
        "--mc-samples",    str(cfg.MC_SAMPLES),
        "--tta",
        "--save-uncertainty",
    ]
    print("Running:", " ".join(infer_cmd))
    subprocess.run(infer_cmd)
else:
    print("ERROR: No checkpoint found. Train first.")



## 6. Local Evaluation

Mirrors the Synapse server evaluation: SSIM, PSNR, MSE on mask-healthy region,
normalised by max(t1n_voided).

In [ ]:
import subprocess, sys

eval_cmd = [
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--max-cases", "30",  # Quick sanity check; remove for full eval
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd)



## 7. Submission Checklist

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--checklist",
])



## 8. Package & Upload Submission

This section:
1. Zips all `*-t1n-inference.nii.gz` predictions into a submission archive
2. Validates the zip contents
3. Uploads to Synapse

In [ ]:
import zipfile
from pathlib import Path

# -- Zip the predictions ------------------------------------------------------
results_dir = cfg.RESULTS_DIR
zip_path = Path("/kaggle/working/submission.zip")

pred_files = sorted(results_dir.glob("BraTS-GLI-*-*-t1n-inference.nii.gz"))
assert len(pred_files) > 0, f"No prediction files found in {results_dir}"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in pred_files:
        # Store with flat name (no subdirectory) as expected by the challenge
        zf.write(f, arcname=f.name)

# -- Validate the zip ---------------------------------------------------------
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print(f"Submission zip: {zip_path}")
    print(f"   Files in archive : {len(names)}")
    print(f"   Archive size     : {zip_path.stat().st_size / 1e6:.1f} MB")
    bad = [n for n in names if not n.endswith("-t1n-inference.nii.gz")]
    if bad:
        print(f"   Unexpected filenames: {bad[:5]}")
    else:
        print(f"   All filenames follow BraTS convention")
    print(f"   Sample: {names[0]}")



### 8.1 Upload to Synapse

Requires your Synapse auth token stored as a Kaggle Secret named `brats_upload`.
To set it up:
1. Go to [Synapse](https://www.synapse.org/) → Profile → Personal Access Tokens
2. Create a token with `view`, `download`, `modify` permissions
3. Add it as a Kaggle Secret: Notebook → Add-ons → Secrets → `brats_upload`

In [ ]:
import synapseclient
from kaggle_secrets import UserSecretsClient

# Retrieve auth token from Kaggle Secrets
secrets = UserSecretsClient()
UPLOAD_TOKEN = secrets.get_secret("brats_upload")

syn_up = synapseclient.Synapse()
syn_up.login(authToken=UPLOAD_TOKEN, silent=True)
print(f"Logged in as: {syn_up.getUserProfile()['userName']}")

# -- Upload submission --------------------------------------------------------
SUBMISSION_FOLDER_ID = "syn75156645"

uploaded = syn_up.store(synapseclient.File(
    path        = str(zip_path),
    parent      = SUBMISSION_FOLDER_ID,
    description = "BraTS inpainting - BrainTart v2 (MonoUNet + MC-Dropout + Edge/Freq loss)",
))

print(f"\n{'='*55}")
print(f"SUBMISSION UPLOADED")
print(f"{'='*55}")
print(f"  File        : {uploaded.name}")
print(f"  Entity ID   : {uploaded.id}")
print(f"  Folder      : {SUBMISSION_FOLDER_ID}")
print(f"  Size        : {zip_path.stat().st_size / 1e6:.1f} MB")
print(f"  Predictions : {len(pred_files)} cases")
print(f"  Method      : MC-Dropout ({cfg.MC_SAMPLES}×) + TTA({'on' if cfg.USE_TTA_FLIPS else 'off'})")
print(f"{'='*55}")



## v2 Improvement Summary

| Feature | v1 | v2 |
|---|---|---|
| **Loss** | L1 + SSIM + DS | L1 + SSIM + **Edge** + **Freq** + DS |
| **Scheduler** | Cosine (cold start) | **Warmup** → Cosine |
| **Batch size** | 2/GPU | 2/GPU × **2 accum** = 4 effective |
| **Inference** | Single forward pass | **MC-Dropout** (8×) + **TTA** (4×) |
| **Uncertainty** | None | **Per-voxel σ** maps |
| **Bottleneck** | ResBlock + Attn + ResBlock | ResBlock + Attn + **Dropout3d** + ResBlock |